In [3]:
from pathlib import Path
import shutil
import pycolmap

import numpy as np
import matplotlib.pyplot as plt

In [4]:
def read_array(path): # read binary files from colmap
    with open(path, "rb") as fid:
        width, height, channels = np.genfromtxt(
            fid, delimiter="&", max_rows=1, usecols=(0, 1, 2), dtype=int
        )
        fid.seek(0)
        num_delimiter = 0
        byte = fid.read(1)
        while True:
            if byte == b"&":
                num_delimiter += 1
                if num_delimiter >= 3:
                    break
            byte = fid.read(1)
        array = np.fromfile(fid, np.float32)
    array = array.reshape((width, height, channels), order="F")
    return np.transpose(array, (1, 0, 2)).squeeze()

In [5]:
output_path      = Path("example/")
image_path       = Path("Horse/")
database_path    = output_path / "database.db"
sfm_path         = output_path / "sfm"
undistorted_path = output_path / "undistorted"

In [4]:
pycolmap.undistort_images(undistorted_path, sfm_path / "0", image_path)

I20250806 19:44:01.273576 140317107573184 images.cc:110]  => Reconstruction with 151 images and 76082 points
I20250806 19:44:01.273764 140317107573184 misc.cc:44] 
Image undistortion
I20250806 19:44:01.280554 140317107573184 undistortion.cc:197] Undistorting image [1/151]
I20250806 19:44:02.536155 140317107573184 undistortion.cc:197] Undistorting image [2/151]
I20250806 19:44:03.318151 140317107573184 undistortion.cc:197] Undistorting image [3/151]
I20250806 19:44:03.318368 140317107573184 undistortion.cc:197] Undistorting image [4/151]
I20250806 19:44:03.323443 140317107573184 undistortion.cc:197] Undistorting image [5/151]
I20250806 19:44:03.323567 140317107573184 undistortion.cc:197] Undistorting image [6/151]
I20250806 19:44:03.323597 140317107573184 undistortion.cc:197] Undistorting image [7/151]
I20250806 19:44:04.184522 140317107573184 undistortion.cc:197] Undistorting image [8/151]
I20250806 19:44:04.184578 140317107573184 undistortion.cc:197] Undistorting image [9/151]
I202508

# IMPORTANT
Patch matching requires colmap compiled with CUDA support!

With the versions available in conda-forge as of now, this requires a computer with cuda 12.6, which is not the case for me.

You may compile colmap with your cuda version yourself, if you follow the instructions [here](https://colmap.github.io/install.html#build-from-source), however, debugging compilation errors is a secret art that I'm not very good at, so I wont be able to help.

When the classes were being recorded, GPU-enabled COLMAP 3.9 was still working for me, but packages were upgraded (critically, hloc and pycolmap), so downgrading COLMAP to 3.9 is not possible. Sorry for that.

In [5]:
pycolmap.patch_match_stereo(undistorted_path)

E20250806 19:44:26.904347 140317107573184 mvs.cc:40] PatchMatch requires CUDA but COLMAP was not compiled with it.


RuntimeError: [mvs.cc:40] PatchMatch requires CUDA but COLMAP was not compiled with it.

In [ ]:
for i, path in enumerate((undistorted_path/"images").glob("*")):
    img = plt.imread(path)
    depth_path = undistorted_path / "stereo" / "depth_maps" / (path.name + ".photometric.bin")
    depth_map  = read_array(depth_path)    

    fig, ax = plt.subplots(1, 2, figsize=(16, 9), sharey=True, constrained_layout=True)
    
    ax[0].imshow(img)
    z = ax[1].imshow(depth_map, vmin=0, vmax=20, cmap="viridis_r")
    cb = fig.colorbar(z, ax=ax[1], shrink=0.4, extend="max")
    cb.set_label("Depth")
    if i > 8:
        break

In [ ]:
lims = 4*np.r_[1, 1, 1]
pycolmap.stereo_fusion(undistorted_path / "dense.ply", undistorted_path, input_type="photometric",
                       options={"bounding_box": (-lims, lims)} )

I20250806 19:45:32.292893 140317107573184 misc.cc:51] 
StereoFusion::Options
---------------------
I20250806 19:45:32.292930 140317107573184 fusion.cc:76] mask_path: 
I20250806 19:45:32.292934 140317107573184 fusion.cc:77] max_image_size: -1
I20250806 19:45:32.292937 140317107573184 fusion.cc:78] min_num_pixels: 5
I20250806 19:45:32.292938 140317107573184 fusion.cc:79] max_num_pixels: 10000
I20250806 19:45:32.292940 140317107573184 fusion.cc:80] max_traversal_depth: 100
I20250806 19:45:32.292942 140317107573184 fusion.cc:81] max_reproj_error: 2
I20250806 19:45:32.292951 140317107573184 fusion.cc:82] max_depth_error: 0.01
I20250806 19:45:32.292955 140317107573184 fusion.cc:83] max_normal_error: 10
I20250806 19:45:32.292958 140317107573184 fusion.cc:84] check_num_images: 50
I20250806 19:45:32.292959 140317107573184 fusion.cc:85] use_cache: 0
I20250806 19:45:32.292961 140317107573184 fusion.cc:86] cache_size: 32
I20250806 19:45:32.292964 140317107573184 fusion.cc:89] bbox_min: -4 -4 -4
I2

In [ ]:
import open3d as o3d
sample_ply_data = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(undistorted_path /"dense.ply")
o3d.visualization.draw_geometries([pcd],
                                  zoom=0.05,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[0, 0, 0],
                                  up=[0, -1, 0])